In [1]:
import os
import zipfile
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import classification_report

In [5]:
folder = "data"
activities = {"walking": "walking","running": "running","idle": "idle","stairs": "stairs"}
frames = []
for label, folder_name in activities.items():
    path = os.path.join(folder, folder_name)
    for file in os.listdir(path):
        if file.endswith(".csv"):
            df = pd.read_csv(os.path.join(path, file))
            df["activity"] = label
            frames.append(df)
data = pd.concat(frames, ignore_index=True)
data.head()

,accelerometer_X,accelerometer_Y,accelerometer_Z,activity
0,-4.965574,-7.340622,0.134075,walking
1,-7.704541,-15.069105,-1.225831,walking
2,-2.676717,-0.857124,7.896077,walking
3,-7.053318,-15.361198,5.477934,walking
4,-7.072471,-10.232818,13.153744,walking


In [8]:
print(data.shape)
print(data.columns)
data["activity"].value_counts()

(193860, 4)
Index(['accelerometer_X', 'accelerometer_Y', 'accelerometer_Z', 'activity'], dtype='object')


,count
activity,
running,102240
walking,55500
idle,31170
stairs,4950


In [9]:
X_raw = data[[
    "accelerometer_X",
    "accelerometer_Y",
    "accelerometer_Z"
]]

y_raw = data["activity"]

In [12]:
feature_data = []
window = 50
for activity in data["activity"].unique():
    activity_df = data[data["activity"] == activity]
    values = activity_df[["accelerometer_X","accelerometer_Y","accelerometer_Z"]].values
    for i in range(0, len(values)-window, window):
        part = values[i:i+window]
        feature_data.append({
            "mean_x": np.mean(part[:,0]),
            "mean_y": np.mean(part[:,1]),
            "mean_z": np.mean(part[:,2]),
            "std_x": np.std(part[:,0]),
            "std_y": np.std(part[:,1]),
            "std_z": np.std(part[:,2]),
            "max_x": np.max(part[:,0]),
            "max_y": np.max(part[:,1]),
            "max_z": np.max(part[:,2]),
            "min_x": np.min(part[:,0]),
            "min_y": np.min(part[:,1]),
            "min_z": np.min(part[:,2]),
            "activity": activity
        })
feature_df = pd.DataFrame(feature_data)
feature_df.head()

,mean_x,mean_y,mean_z,std_x,std_y,std_z,max_x,max_y,max_z,min_x,min_y,min_z,activity
0,-2.513146,-10.744315,-0.494163,4.777608,6.358152,6.422489,11.894394,9.351752,24.660278,-12.277467,-23.673866,-16.338032,walking
1,-1.309820,-10.196426,-1.127477,4.636058,7.169687,7.467784,8.475474,7.809886,21.648373,-11.669339,-27.911604,-26.302700,walking
2,-3.836660,-10.056126,-0.837587,4.745336,6.424852,8.931400,6.378154,9.270349,32.728737,-15.777789,-23.376986,-21.021091,walking
3,-3.992283,-9.436219,-1.707353,3.817304,4.904351,6.400931,1.982399,4.783615,23.467964,-16.333244,-21.356280,-16.127342,walking
4,-3.238972,-8.403744,-0.197378,3.597731,5.651403,8.057166,7.580043,6.770803,34.261024,-13.953407,-19.048270,-17.142485,walking


In [14]:
X_feat = feature_df.drop("activity", axis=1)
y_feat = feature_df["activity"]

In [17]:
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(X_raw,y_raw,test_size=0.2,random_state=42)
scaler = StandardScaler()
X_train_raw = scaler.fit_transform(X_train_raw)
X_test_raw = scaler.transform(X_test_raw)

In [16]:
X_train_feat, X_test_feat, y_train_feat, y_test_feat = train_test_split(X_feat,y_feat,test_size=0.2,random_state=42)
scaler2 = StandardScaler()
X_train_feat = scaler2.fit_transform(X_train_feat)
X_test_feat = scaler2.transform(X_test_feat)

In [18]:
svm_raw = SVC()
svm_raw.fit(X_train_raw, y_train_raw)
pred = svm_raw.predict(X_test_raw)
print(classification_report(y_test_raw, pred))

              precision    recall  f1-score   support

        idle       0.96      0.98      0.97      6221
     running       0.93      0.91      0.92     20533
      stairs       1.00      0.00      0.01       915
     walking       0.81      0.90      0.85     11103

    accuracy                           0.90     38772
   macro avg       0.92      0.70      0.69     38772
weighted avg       0.90      0.90      0.89     38772



Модель показала загальну точність близько 90% добре класифікуючи класи idle running та walking проте клас stairs практично не був розпізнаний про що свідчить Recall рівний 0 та F1-score 0.01.

In [19]:
rf_raw = RandomForestClassifier(random_state=42)
rf_raw.fit(X_train_raw, y_train_raw)
pred = rf_raw.predict(X_test_raw)
print(classification_report(y_test_raw, pred))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00      6221
     running       1.00      1.00      1.00     20533
      stairs       1.00      0.99      1.00       915
     walking       1.00      1.00      1.00     11103

    accuracy                           1.00     38772
   macro avg       1.00      1.00      1.00     38772
weighted avg       1.00      1.00      1.00     38772



Модель Random Forest продемонструвала майже ідеальні результати для всіх класів значення Precision Recall та F1-score практично дорівнюють 1 що свідчить про дуже якісну класифікацію.

In [20]:
svm_feat = SVC()
svm_feat.fit(X_train_feat, y_train_feat)
pred = svm_feat.predict(X_test_feat)
print(classification_report(y_test_feat, pred))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       130
     running       1.00      1.00      1.00       394
      stairs       0.94      0.84      0.89        19
     walking       0.99      1.00      0.99       232

    accuracy                           0.99       775
   macro avg       0.98      0.96      0.97       775
weighted avg       0.99      0.99      0.99       775



Після переходу до часових характеристик якість моделі суттєво покращилася особливо для класу stairs де Recall збільшився з 0 до 0.84 а F1-score до 0.89.

In [21]:
rf_feat = RandomForestClassifier(random_state=42)
rf_feat.fit(X_train_feat, y_train_feat)
pred = rf_feat.predict(X_test_feat)
print(classification_report(y_test_feat, pred))

              precision    recall  f1-score   support

        idle       1.00      1.00      1.00       130
     running       1.00      1.00      1.00       394
      stairs       1.00      0.89      0.94        19
     walking       0.99      1.00      1.00       232

    accuracy                           1.00       775
   macro avg       1.00      0.97      0.99       775
weighted avg       1.00      1.00      1.00       775



Random Forest також показав дуже високі результати після використання часових ознак для більшості класів усі основні метрики дорівнюють 1 а для класу stairs Recall зріс до 0.89 що є кращим результатом порівняно із SVM